# Notebook 5: SHAP Values & Isolation Forests

**Series:** Random Forests & Isolation Forests for Full-Stack Engineers  
**Prerequisites:** Notebooks 1–4  
**Author:** [Farty Bobo](https://fartybobo.com)
**What you'll learn:**
1. SHAP values: explaining WHY the model predicted what it did for a specific input
2. Isolation Forests: using tree methods for anomaly/outlier detection

---

## Part 1: SHAP Values — Explainability

### The Problem

You've trained a Random Forest. It has 90% accuracy. Great. But your manager asks:
> "Why did the model flag this specific user as fraudulent?"

Feature importance (Notebook 4) tells you which features matter *in general* across the whole model. But it doesn't tell you: *for this one prediction, which features pushed the model toward this outcome?*

That's what **SHAP values** are for.

---

### The Game Theory Analogy (Shapley Values)

SHAP stands for **SH**apley **A**dditive ex**P**lanations. The underlying math (Shapley values) comes from cooperative game theory.

The question in game theory: if a team of players wins a prize together, how do you fairly distribute credit among them?

The Shapley answer: each player's fair share = their **average marginal contribution** — how much they contributed across all possible orderings of when players join the team.

In ML, players = features. Prize = model's prediction. SHAP asks: **what's each feature's fair contribution to this specific prediction?**

```
prediction = baseline_prediction + SHAP(feature_1) + SHAP(feature_2) + ... + SHAP(feature_n)
```

The baseline is the average prediction across all training data. Each SHAP value is the amount that feature *shifted* the prediction from baseline — positive means "pushed toward this class", negative means "pushed away".

---

### The Engineering Analogy

Think of it like profiling a slow API request. You know the total response time. But which middleware added the most latency? You break the total latency into per-component contributions:

```
Total: 450ms
  Auth middleware:  +20ms  (pushed latency up by 20ms)
  DB query:        +380ms  (main culprit — huge positive contribution)
  Cache check:      -30ms  (reduced latency — negative contribution)
  Serialization:    +80ms
```

SHAP values do the same for predictions:
```
P(virginica) = 0.12 (baseline)
  petal_width:  +0.65  (pushed strongly toward virginica)
  petal_length: +0.12  (small positive push)
  sepal_length: -0.04  (slight push away)
  sepal_width:  -0.01  (negligible)
final P(virginica) = 0.84
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

pal = sns.color_palette('colorblind')
sns.set_palette(pal)

# shap needs to be installed: uv add shap
try:
    import shap
    shap.initjs()  # initializes JavaScript for interactive plots in Jupyter
    SHAP_AVAILABLE = True
    print('shap is available!')
except ImportError:
    SHAP_AVAILABLE = False
    print('shap not installed. Run: uv add shap')
    print('The code cells below will show expected output in comments.')

iris = load_iris()

# Wrap as DataFrame for SHAP (it needs named columns)
X_df = pd.DataFrame(iris.data, columns=iris.feature_names)
y    = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

print(f'Test accuracy: {rf.score(X_test, y_test):.1%}')

In [ ]:
# Compute SHAP values using the TreeExplainer
# TreeExplainer is optimized for tree-based models (Random Forest, XGBoost, etc.)
# It's MUCH faster than the generic KernelExplainer.

if SHAP_AVAILABLE:
    explainer  = shap.TreeExplainer(rf)
    shap_values = explainer(X_test)  # shape: (n_test, n_features, n_classes)
    
    print(f'SHAP values shape: {shap_values.values.shape}')
    print(f'  → {shap_values.values.shape[0]} test samples')
    print(f'  → {shap_values.values.shape[1]} features')
    print(f'  → {shap_values.values.shape[2]} classes')
    print()
    print('SHAP values for sample 0, class "virginica" (index 2):')
    for feat, val in zip(iris.feature_names, shap_values.values[0, :, 2]):
        direction = '↑' if val > 0 else '↓'
        print(f'  {feat:25s}: {direction} {val:+.4f}')
    print(f'  Base value (average pred): {shap_values.base_values[0, 2]:.4f}')
    print(f'  Sum = {shap_values.base_values[0,2] + shap_values.values[0,:,2].sum():.4f}  ← should match predict_proba')
    print(f'  predict_proba result:      {rf.predict_proba(X_test.iloc[[0]])[0,2]:.4f}')
else:
    print('Install shap to run this cell: uv add shap')

In [ ]:
# === Waterfall Plot: Explain a SINGLE prediction ===
# This is the "profiler" view — for one sample, shows which features
# pushed the prediction up or down.

if SHAP_AVAILABLE:
    # Focus on predicting class "versicolor" (index 1)
    class_idx = 1
    class_name = iris.target_names[class_idx]
    
    # Build an Explanation object for the class we care about
    exp = shap.Explanation(
        values      = shap_values.values[:, :, class_idx],
        base_values = shap_values.base_values[:, class_idx],
        data        = X_test.values,
        feature_names = X_test.columns.tolist()
    )
    
    # Plot the first 3 test samples
    for idx in range(3):
        true_label = iris.target_names[y_test.iloc[idx]]
        pred_label = iris.target_names[rf.predict(X_test.iloc[[idx]])[0]]
        print(f'Sample {idx}: true={true_label}, predicted={pred_label}')
        shap.plots.waterfall(exp[idx])
else:
    print('Install shap to see waterfall plots: uv add shap')
    print()
    print('Expected output: a bar chart showing how each feature shifts the')
    print('prediction from the base rate (average across all samples) to')
    print('the final predicted probability. Red bars push UP, blue bars push DOWN.')

In [ ]:
# === Summary Plot: Feature importance across ALL test samples ===
# Shows both direction and magnitude of each feature across the whole test set.
# Color = feature value (red=high, blue=low), x-axis = SHAP impact.

if SHAP_AVAILABLE:
    print('Global SHAP summary plot — shows feature importance AND directionality:')
    # For multi-class, SHAP returns values per class. We'll look at all classes.
    shap.summary_plot(
        shap_values.values,    # shape: (n_samples, n_features, n_classes)
        X_test.values,
        plot_type='bar',
        class_names=iris.target_names,
        feature_names=X_test.columns.tolist()
    )
else:
    print('Install shap: uv add shap')

In [ ]:
# === Dependence Plot: How does one feature affect predictions? ===
# Shows each sample's SHAP value for one feature, plotted against the feature's value.
# Color = a second feature (auto-selected for the most interaction).
# Reveals: is the relationship linear? Non-monotonic? Threshold-based?

if SHAP_AVAILABLE:
    # Focus on 'petal width (cm)' for each class
    feature_to_examine = 'petal width (cm)'
    feature_idx = X_test.columns.tolist().index(feature_to_examine)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), dpi=100)
    
    for class_idx, class_name in enumerate(iris.target_names):
        shap.dependence_plot(
            feature_to_examine,
            shap_values.values[:, :, class_idx],
            X_test.values,
            feature_names=X_test.columns.tolist(),
            ax=axes[class_idx],
            show=False
        )
        axes[class_idx].set_title(f'Class: {class_name}', fontsize=11)
    
    plt.suptitle(f'How "{feature_to_examine}" Impacts Predictions Per Class', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Install shap: uv add shap')
    print()
    print('Expected: 3 scatter plots (one per class).')
    print('x-axis = petal width value, y-axis = SHAP contribution to that class.')
    print('For setosa: negative SHAP everywhere (wide petals → AWAY from setosa)')
    print('For virginica: positive SHAP for wide petals (wide petals → TOWARD virginica)')

### A Note on TreeExplainer vs KernelExplainer

Tom McT's original notebook used `KernelExplainer`, which is model-agnostic (works with any model, including linear models, neural nets, etc.) but very slow.

`TreeExplainer` is specifically optimized for tree-based models and is much faster with exact (not approximate) SHAP values. Always use `TreeExplainer` with Random Forests, XGBoost, LightGBM, etc.

---

## Part 2: Isolation Forests — Anomaly Detection

### The Problem

So far we've done **supervised learning** — we have labels (setosa/versicolor/virginica) and teach the model to predict them.

But what if you just want to find **weird outliers** without labels? 
- Which transactions are anomalous?
- Which server metrics indicate unusual behavior?
- Which user sessions look like bots?

This is **unsupervised anomaly detection** — no labels required.

---

### The Intuition: Anomalies Are Easy to Isolate

Key insight from the [original Isolation Forest paper](https://cs.nju.edu.cn/zhouzh/zhouzh.files/publication/tkdd11.pdf): **anomalies are few and different**. This means they're easier to isolate than normal points.

Imagine playing 20 Questions to isolate one specific data point:
- If the point is a **normal** (common) data point, it's surrounded by similar points. It takes MANY questions to isolate it.
- If the point is an **anomaly** (rare, different), it's alone in feature space. Just a few questions isolate it.

The Isolation Forest uses random binary splits (like decision trees) to literally "isolate" each point. Points that get isolated quickly = anomalies.

```
Algorithm:
1. For each of n_estimators trees:
   a. Randomly sample a subset of data points
   b. Build a tree by: randomly pick a feature, randomly pick a split value,
      recurse until each sample is isolated in its own leaf
2. For each sample, average the "depth" it reached across all trees
   Short average depth = anomaly (isolated quickly)
   Deep average depth  = normal (hard to isolate)
```

The key differences from a Random Forest classifier:
- **Unsupervised:** no labels needed
- **No optimization objective:** splits are purely random (no Gini minimization)
- **Score is isolation depth**, not class probability

In [ ]:
from sklearn.ensemble import IsolationForest
from matplotlib.colors import ListedColormap

# Create a 2D dataset: mostly normal data with a few clear outliers
np.random.seed(42)

n_normal  = 300
n_outlier = 20

# Normal data: two clusters
X_normal_a = np.random.randn(n_normal // 2, 2) * 0.5 + [1, 1]   # cluster near (1,1)
X_normal_b = np.random.randn(n_normal // 2, 2) * 0.5 + [-1, -1] # cluster near (-1,-1)
X_normal   = np.vstack([X_normal_a, X_normal_b])

# Outliers: scattered far from the clusters
X_outliers = np.random.uniform(-5, 5, size=(n_outlier, 2))
# Filter to ones that are actually far from the clusters
dist_from_center = np.sqrt((X_outliers**2).sum(axis=1))
X_outliers = X_outliers[dist_from_center > 3][:n_outlier]

X_all  = np.vstack([X_normal, X_outliers])
labels = np.array([1] * len(X_normal) + [-1] * len(X_outliers))  # 1=normal, -1=outlier

print(f'Normal samples: {len(X_normal)}')
print(f'Outlier samples: {len(X_outliers)}')
print(f'Contamination rate: {len(X_outliers)/len(X_all):.1%}')

In [ ]:
# Train the Isolation Forest
# contamination = fraction of the dataset you expect to be outliers
# This sets the threshold for the anomaly score

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=len(X_outliers)/len(X_all),  # tell it what fraction are anomalies
    random_state=42
)

# fit_predict returns 1 (normal) or -1 (anomaly) for each point
y_pred = iso_forest.fit_predict(X_all)

# Anomaly score: more negative = more anomalous
# decision_function returns the raw score before thresholding
anomaly_scores = iso_forest.decision_function(X_all)

print(f'Points classified as normal:  {(y_pred == 1).sum()}')
print(f'Points classified as outlier: {(y_pred == -1).sum()}')

# Compute confusion: did we catch the actual outliers?
true_positives  = ((y_pred == -1) & (labels == -1)).sum()
false_positives = ((y_pred == -1) & (labels == 1)).sum()
false_negatives = ((y_pred == 1) & (labels == -1)).sum()

print(f'\nOutlier Detection:')
print(f'  True positives  (correctly flagged outliers): {true_positives}')
print(f'  False positives (normal points flagged):      {false_positives}')
print(f'  False negatives (missed outliers):            {false_negatives}')

In [ ]:
# Visualize: show the data, the model's decision boundary, and the anomaly scores

h = 0.05  # mesh resolution
x_min, x_max = X_all[:, 0].min() - 1, X_all[:, 0].max() + 1
y_min, y_max = X_all[:, 1].min() - 1, X_all[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z_score = iso_forest.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
Z_pred  = iso_forest.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

# --- Left: Decision boundary ---
ax = axes[0]
ax.contourf(xx, yy, Z_score, levels=50, cmap='RdYlGn', alpha=0.7)
ax.contour(xx, yy, Z_pred, levels=[0], linewidths=2, colors='black')

# Color by TRUE label
scatter_normal = ax.scatter(X_normal[:, 0], X_normal[:, 1], c='steelblue', s=15, 
                             label='Normal (true)', edgecolors='k', lw=0.3)
scatter_outlier = ax.scatter(X_outliers[:, 0], X_outliers[:, 1], c='orangered', s=60, 
                              marker='*', label='Outlier (true)', edgecolors='k', lw=0.5)
ax.set_title('Isolation Forest: Score Map\n(green=normal region, red=anomaly region)', fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(Z_score.min(), Z_score.max()))
plt.colorbar(sm, ax=ax, label='Anomaly score (more negative = more anomalous)')

# --- Right: Predicted vs actual ---
ax2 = axes[1]
colors = np.where(y_pred == 1, 'steelblue', 'orangered')
ax2.scatter(X_all[:, 0], X_all[:, 1], c=colors, s=20, edgecolors='k', lw=0.3)

# Mark mistakes
mistakes = y_pred != labels
ax2.scatter(X_all[mistakes, 0], X_all[mistakes, 1], s=150, marker='x', 
            c='black', linewidth=2, label='Misclassified')
ax2.set_title('Predicted Labels\n(blue=predicted normal, red=predicted anomaly, X=mistakes)', fontsize=11)
ax2.legend(fontsize=9)
ax2.set_xlim(x_min, x_max); ax2.set_ylim(y_min, y_max)

plt.suptitle('Isolation Forest on Synthetic Dataset', fontsize=13)
plt.tight_layout()
plt.show()

### Key Isolation Forest Parameters

| Parameter | Default | What it controls |
|---|---|---|
| `n_estimators` | 100 | Number of isolation trees |
| `max_samples` | `'auto'` (256) | Samples per tree — smaller = faster but noisier |
| `contamination` | `'auto'` | Expected fraction of outliers — sets decision threshold |
| `max_features` | 1.0 | Features per tree (fraction) |

`contamination` is the most important tuning parameter. If you set it too low, you miss real anomalies. Too high, you flag normal things as anomalous. Use domain knowledge to set this.

---

## ExtraTrees: A Quick Note

There's a variant of Random Forest called **ExtraTrees** (Extremely Randomized Trees, `ExtraTreesClassifier`). The difference:

- **Random Forest:** picks the BEST split threshold for each feature subset
- **ExtraTrees:** picks a RANDOM threshold for each feature (even more random!)

ExtraTrees is often faster to train and performs comparably or better. The trade-off: it has higher bias per tree, but even more variance reduction via randomization.

**Practical note:** In terms of API, they're identical in sklearn. Just swap `RandomForestClassifier` → `ExtraTreesClassifier`. The hyperparameters are the same. Try both, keep whichever scores better.

**Production note:** Not all platforms support ExtraTrees. For example, PySpark ML supports `RandomForest` but not `ExtraTrees`. If you're deploying to a constrained platform, check support before committing to ExtraTrees.

---

In [ ]:
# Quick comparison: RandomForest vs ExtraTrees on Iris
from sklearn.ensemble import ExtraTreesClassifier

iris = load_iris()
X_tr, X_te, y_tr, y_te = train_test_split(iris.data, iris.target, test_size=0.2, 
                                           random_state=42, stratify=iris.target)

models_compare = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'ExtraTrees':   ExtraTreesClassifier(n_estimators=100, random_state=42),
}

import time
print(f'{"Model":<20} {"Train Acc":>10} {"Test Acc":>10} {"Train Time":>12}')
print('-' * 55)
for name, model in models_compare.items():
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    train_time = time.perf_counter() - t0
    print(f'{name:<20} {model.score(X_tr, y_tr):>10.1%} {model.score(X_te, y_te):>10.1%} {train_time*1000:>10.1f}ms')

---

## Summary: The Full Series

Congratulations — you've gone from zero ML knowledge to understanding the full Random Forest ecosystem.

Here's your mental map:

```
CLASSIFICATION PROBLEM
         │
         ▼
   Prepare Data
   ├── Handle NULLs (median impute or out-of-range sentinel)
   ├── Transform distributions (QuantileTransformer)
   └── Train/test split (NEVER leak test data)
         │
         ▼
   Build Ensemble
   ├── Many Decision Trees (if/else rules learned from data)
   ├── Each tree: bootstrap sample + random feature subsets
   └── Final prediction = majority vote (or averaged probabilities)
         │
         ▼
   Evaluate & Explain
   ├── Test accuracy (always on held-out data)
   ├── Feature importance (permutation-based, not impurity-based)
   └── SHAP values (per-prediction explanations)

ANOMALY DETECTION (no labels needed)
         │
         ▼
   Isolation Forest
   ├── Random trees isolate each point
   ├── Anomalies = isolated quickly (short average depth)
   └── Score + contamination threshold → flag outliers
```

---

## What to Try Next

1. **Use your own data.** Replace the Iris dataset with a CSV from your work. Everything else stays the same.
2. **Tune hyperparameters.** Try `GridSearchCV` or `RandomizedSearchCV` to automatically find good `max_depth`, `n_estimators`, etc.
3. **Try XGBoost/LightGBM.** These are gradient boosting methods (sequential trees, not parallel) that often outperform Random Forests on tabular data.
4. **SHAP in production.** Use SHAP values in your API responses to explain why the model flagged a specific item.

---

| Concept | Notebook |
|---|---|
| Classification + Decision Trees + Gini Impurity | 01 |
| Feature distributions + QuantileTransformer + Train/test split | 02 |
| Random Forests + Bootstrap + Ensemble diversity | 03 |
| NULL handling + Feature importance | 04 |
| SHAP values + Isolation Forests + ExtraTrees | 05 |